# 16 — Perceptron

**Algorithm:** Rosenblatt Perceptron (single-layer)  
**Features:** TF-IDF (50K unigrams + bigrams)  
The Perceptron is the simplest online linear classifier. Its convergence is guaranteed only for linearly separable data, making the per-epoch accuracy plot particularly revealing.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, evaluate_and_print, plot_confusion_matrix,
                   plot_roc_curve, plot_pr_curve)
from sklearn.metrics import accuracy_score
BASE_DIR, FIGURES_DIR = setup_paths()

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), sublinear_tf=True, min_df=2, max_df=0.95)
X_tr = tfidf.fit_transform(X_train)
X_te = tfidf.transform(X_test)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## Training Convergence via partial_fit

In [ ]:
from sklearn.linear_model import Perceptron
classes = np.unique(y_train)
n_epochs = 20
batch_size = 2048
n = X_tr.shape[0]
tr_accs, te_accs = [], []
perc = Perceptron(max_iter=1, random_state=42, n_jobs=-1)
for epoch in range(1, n_epochs + 1):
    idx = np.random.permutation(n)
    for start in range(0, n, batch_size):
        batch = idx[start:start + batch_size]
        perc.partial_fit(X_tr[batch], y_train[batch], classes=classes)
    tr_accs.append(accuracy_score(y_train, perc.predict(X_tr)))
    te_accs.append(accuracy_score(y_test,  perc.predict(X_te)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, n_epochs+1), tr_accs, 'o-', color='#3498db', lw=2, ms=5, label='Train Accuracy')
ax.plot(range(1, n_epochs+1), te_accs, 's-', color='#e74c3c', lw=2, ms=5, label='Test Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Perceptron — Training Convergence (online updates)')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'perceptron_convergence.png'), dpi=150, bbox_inches='tight')
plt.show()

## Final Model

In [ ]:
MODEL_NAME = 'Perceptron'
t0 = time.time()
model = Perceptron(max_iter=1000, random_state=42, n_jobs=-1)
model.fit(X_tr, y_train)
train_time = time.time() - t0
metrics, y_pred, y_prob = evaluate_and_print(model, X_te, y_test, MODEL_NAME, train_time)

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)